In [0]:
INSERT INTO prod.detection.{golden_table_cfe_merge_modularized}
WITH clients_table_to_join AS (
  SELECT *
  FROM prod.detection.clients
  WHERE client_name IN ('kinetiq', 'SpringServe-Prod')
)
, nielsen_replacement_national_nyc_alias AS (
    SELECT rl.station_id, rl.fk_show_id, rl.tuner_channel_id, rl.tuner_program_id
    FROM prod.detection.nielsen_replacement_national_nyc AS rl
    JOIN prod.detection.nielsen_only_distribution_blacklist AS bl
      ON bl.station_id = rl.station_id
     AND bl.blacklist_end >= '{start_time}'
    GROUP BY 1,2,3,4
)
, nielsen_replacement_local_alias AS (
    SELECT rl.station_id, rl.fk_show_id, rl.dma_id, rl.tuner_channel_id, rl.tuner_program_id
    FROM prod.detection.nielsen_replacement_local AS rl
    JOIN prod.detection.nielsen_only_distribution_blacklist AS bl
      ON bl.station_id = rl.station_id
     AND bl.blacklist_end >= '{start_time}'
    GROUP BY 1,2,3,4,5
)
, activity_obfuscation AS (
  SELECT blocked_apps.app_name, override.client_name
  FROM prod.detection.app_activity_distribution_blacklist AS blocked_apps
  LEFT JOIN prod.detection.app_customer_activity_distribution_override override
    ON blocked_apps.app_name = override.app_name
  GROUP BY 1, 2
)
, viewing_obfuscation AS (
  SELECT blocked_apps.app_name, override.client_name
  FROM prod.detection.app_viewing_distribution_blacklist AS blocked_apps
  LEFT JOIN prod.detection.app_customer_viewing_distribution_override override
      ON blocked_apps.app_name = override.app_name
  GROUP BY 1, 2 
)
, epg_program_aggregate AS (
  SELECT DISTINCT *
  FROM prod.detection.vizio_epg_program_aggregate
)
, viewing_content_firehose AS (
  SELECT DISTINCT fk_tvid, session_start, session_end, fk_content_id, is_live
  FROM prod.detection.viewing_content_firehose AS content
  WHERE content.session_start >= '{start_time}'::timestamp
      AND content.session_start < '{end_time}'::timestamp
      AND content.partition_key >= '{start_time}'::timestamp::DATE
      AND content.partition_key <= '{end_time}'::timestamp::DATE
)
, content_ids_firehose AS (
  SELECT * FROM detection.content_ids_firehose AS cid
  WHERE content_id IN (
      SELECT DISTINCT c.fk_content_id
      FROM viewing_content_firehose AS c
  )
)
, station_distribution_blacklist AS (
  SELECT vendor_station_id AS station_id, vendor_name, client_name
  FROM prod.detection.station_distribution_obfuscation_overwrite
  GROUP BY 1, 2, 3
)
, inscape_map_deduped AS (
    SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id
    FROM (
        SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id, ROW_NUMBER() OVER (PARTITION BY mapped_vendor, mapped_vendor_station_id ORDER BY created_at DESC) AS rn
        FROM prod.detection.inscape_station_map) ism
    WHERE ism.rn = 1
)
, new_comm_id_ext_firehose AS (
  WITH spring_serve_ads AS (
    WITH fileingest AS (
      SELECT ssm.reportingid AS ssl_id
        , ssm.brand AS brand_name
        , ssm.advertiser AS advertiser
        , CASE WHEN ssm.contenttitle IN ('SpringServe commercial', '[TBD]') THEN NULL ELSE ssm.contenttitle END AS title
        , ssm.content_duration AS duration
        , 1 AS rn
        FROM prod.public.fileingest_cidmap ssm
        WHERE client_id = 'SpringServe-Prod'
          AND brand NOT IN ('SpringServe', '-', '_', '[TBD]')
        GROUP BY 1,2,3,4,5
    )
    , mturk AS (
      SELECT ssm.vast_hash AS ssl_id
        , ssm.advertiser AS brand_name
        , ssm.advertiser AS advertiser
        , ssm.brand AS title
        , ssm.duration
        , 2 AS rn
        FROM prod.public.springserve_metadata ssm
        WHERE ssm.advertiser NOT IN ('', 'SpringServe', '-', '_', '[TBD]')
        GROUP BY 1,2,3,4,5
    )
    SELECT ssl_id, brand_name, advertiser, title, duration
    FROM (
      SELECT *
      , ROW_NUMBER() OVER (PARTITION BY ssl_id ORDER BY rn) AS nrn
      FROM (
        SELECT * FROM fileingest
        UNION
        SELECT * FROM mturk
      ) a
    ) a
    WHERE nrn = 1
  )
  SELECT external_id, brand_name, title, duration
  FROM (
    SELECT external_id
    , brand_name
    , title
    , duration
    , ROW_NUMBER() OVER (PARTITION BY external_id ORDER BY rn) AS new_rn
    FROM (
      SELECT external_id, brand_name, title, duration, 0 as rn
      FROM (
        SELECT *
        , ROW_NUMBER() OVER (PARTITION BY m.external_id ORDER BY if(m.source = 'Ingested',0,1),m.fk_commercial_id DESC) AS rk
        FROM prod.detection.commercial_id_external_firehose AS m
        JOIN clients_table_to_join cl
          ON m.fk_client_id = cl.client_id
        WHERE m.brand_name != 'SpringServe'
          AND m.brand_name IS NOT NULL
      )
      WHERE rk = 1
      UNION ALL
      SELECT ssl_id, brand_name, title, duration, 1 AS rn
      FROM spring_serve_ads
    ) a
  ) 
  WHERE new_rn = 1
)
SELECT tvid, fk_tvid, zipcode, dma, external_id, mt_start, session_start, session_end
, NULLIF(tms_prev_episode_id, '') AS tms_prev_episode_id
, NULLIF(tivo_prev_episode_id, '') AS tivo_prev_episode_id
, NULLIF(tms_next_episode_id, '') AS tms_next_episode_id
, NULLIF(tivo_next_episode_id, '') AS tivo_next_episode_id
, tms_prev_show_title, tivo_prev_show_title, tms_next_show_title, tivo_next_show_title
, tms_prev_callsign, tivo_prev_callsign, tms_next_callsign, tivo_next_callsign
, tms_prev_network_affiliate, tivo_prev_network_affiliate, tms_next_network_affiliate, tivo_next_network_affiliate
, live
, app_service
, prev_ts_start, prev_ts_end, next_ts_start, next_ts_end
, ip, input_category, input_device, brand_name, title, duration
, fk_commercial_source_id
, prev_nielsen_exclusive, next_nielsen_exclusive
, '|'||array_join(collect_set(acrb_client), '|')||'|' AS acrb_clients
, '|'||array_join(collect_set(appb_client), '|')||'|' AS appb_clients
, '|'||array_join(collect_set(prev_station_blacklist_client), '|')||'|' AS prev_station_blacklist_clients
, '|'||array_join(collect_set(next_station_blacklist_client), '|')||'|' AS next_station_blacklist_clients
FROM (
  SELECT DISTINCT COALESCE(tv.long_tvid, tv.vizio_tvid) AS tvid
  , c.fk_tvid
  , NULLIF(location.zipcode, '') AS zipcode
  , REPLACE(dma.dma_name, ',', '') AS dma
  , m.external_id
  , c.media_time_start AS mt_start
  , c.session_start
  , c.session_end
    ------------------ Episode ID -------------------
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN COALESCE(c.tms_prev_station_id, c.prev_station_id) = 0 THEN coalesce(prev_filecontent.external_id, SPLIT(prev_cid.content_cid, '_')[0])
         WHEN prev_chanb.channel_name IS NOT NULL OR c.prev_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN tms_prev_program.database_key IS NOT NULL THEN tms_prev_program.database_key
         WHEN prev_vizio_program.program_tms_id IS NOT NULL THEN prev_vizio_program.program_tms_id
    END AS tms_prev_episode_id
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN COALESCE(c.prev_station_id, c.tms_prev_station_id) = 0 THEN coalesce(prev_filecontent.external_id, SPLIT(prev_cid.content_cid, '_')[0])
         WHEN prev_chanb.channel_name IS NOT NULL OR c.prev_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN tivo_prev_program.database_key IS NOT NULL THEN tivo_prev_program.database_key
    END AS tivo_prev_episode_id
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN COALESCE(c.tms_next_station_id, c.next_station_id) = 0 THEN coalesce(next_filecontent.external_id,SPLIT(next_cid.content_cid, '_')[0])
         WHEN next_chanb.channel_name IS NOT NULL OR c.next_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN tms_next_program.database_key IS NOT NULL THEN tms_next_program.database_key
         WHEN next_vizio_program.program_tms_id IS NOT NULL THEN next_vizio_program.program_tms_id
    END AS tms_next_episode_id
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN COALESCE(c.next_station_id, c.tms_next_station_id) = 0 THEN coalesce(next_filecontent.external_id,SPLIT(next_cid.content_cid, '_')[0])
         WHEN next_chanb.channel_name IS NOT NULL OR c.next_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN tivo_next_program.database_key IS NOT NULL THEN tivo_next_program.database_key
    END AS tivo_next_episode_id
  ------------------ Show Title -------------------
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN COALESCE(c.prev_station_id, c.tms_prev_station_id) = 0 THEN prev_filecontent.title
         WHEN prev_chanb.channel_name IS NOT NULL OR c.prev_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN tms_prev_program.title IS NOT NULL THEN REPLACE(tms_prev_program.title, ',', '')
         WHEN prev_vizio_program.series_aggregate_title IS NOT NULL AND prev_vizio_program.series_aggregate_title != '' THEN REPLACE(prev_vizio_program.series_aggregate_title, ',', '')
         ELSE REPLACE(prev_vizio_program.title, ',', '')
    END AS tms_prev_show_title
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN COALESCE(c.prev_station_id, c.tms_prev_station_id) = 0 THEN prev_filecontent.title
         WHEN prev_chanb.channel_name IS NOT NULL OR c.prev_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN tivo_prev_program.title IS NOT NULL THEN REPLACE(tivo_prev_program.title, ',', '')
         WHEN prev_vizio_program.series_aggregate_title IS NOT NULL AND prev_vizio_program.series_aggregate_title != '' THEN REPLACE(prev_vizio_program.series_aggregate_title, ',', '')
         ELSE REPLACE(prev_vizio_program.title, ',', '')
    END AS tivo_prev_show_title
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN COALESCE(c.next_station_id, c.tms_next_station_id) = 0 THEN next_filecontent.title
         WHEN next_chanb.channel_name IS NOT NULL OR c.next_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN tms_next_program.title IS NOT NULL THEN REPLACE(tms_next_program.title, ',', '')
         WHEN next_vizio_program.series_aggregate_title IS NOT NULL AND next_vizio_program.series_aggregate_title != '' THEN REPLACE(next_vizio_program.series_aggregate_title, ',', '')
         ELSE REPLACE(next_vizio_program.title, ',', '')
    END AS tms_next_show_title
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN COALESCE(c.next_station_id, c.tms_next_station_id) = 0 THEN next_filecontent.title
         WHEN next_chanb.channel_name IS NOT NULL OR c.next_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN tivo_next_program.title IS NOT NULL THEN REPLACE(tivo_next_program.title, ',', '')
         WHEN next_vizio_program.series_aggregate_title IS NOT NULL AND next_vizio_program.series_aggregate_title != '' THEN REPLACE(next_vizio_program.series_aggregate_title, ',', '')
         ELSE REPLACE(next_vizio_program.title, ',', '')
    END AS tivo_next_show_title
  ------------------ Channel Call Sign -------------------
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN tms_prev_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.tms_prev_station_id IS NOT NULL THEN tms_prev_map.inscape_call_sign
         WHEN c.tms_prev_station_id = 0 THEN COALESCE(SPLIT(prev_cid.content_cid, '_')[1],'FILE')
    END AS tms_prev_callsign
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN tivo_prev_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.prev_station_id IS NOT NULL THEN tivo_prev_map.inscape_call_sign
         WHEN c.prev_station_id = 0 THEN COALESCE(SPLIT(prev_cid.content_cid, '_')[1],'FILE')
    END AS tivo_prev_callsign
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN tms_next_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.tms_next_station_id IS NOT NULL THEN tms_next_map.inscape_call_sign
         WHEN c.tms_next_station_id = 0 THEN COALESCE(SPLIT(next_cid.content_cid, '_')[1],'FILE')
    END AS tms_next_callsign
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN tivo_next_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.next_station_id IS NOT NULL THEN tivo_next_map.inscape_call_sign
         WHEN c.next_station_id = 0 THEN COALESCE(SPLIT(next_cid.content_cid, '_')[1],'FILE')
    END AS tivo_next_callsign
  ------------------ Station Affiliate -----------------------
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN tms_prev_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.tms_prev_station_id IS NOT NULL THEN
               CASE WHEN tms_prev_station.inscape_station_name IS NOT NULL THEN tms_prev_station.inscape_station_name
                    WHEN LOWER(tms_prev_station.station_affil) LIKE '%affiliate%'
                          OR LOWER(tms_prev_station.station_affil) LIKE '%independent%'
                          OR LOWER(tms_prev_station.station_affil) LIKE '%low power%' THEN tms_prev_station.station_affil END
         WHEN prev_chanb.channel_name IS NOT NULL OR c.prev_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN 'OBFUSCATED'
         WHEN c.prev_vizio_epg_station IS NOT NULL THEN prev_vizio_station.name
    END AS tms_prev_network_affiliate
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN tivo_prev_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.prev_station_id IS NOT NULL THEN
               CASE WHEN tivo_prev_station.inscape_station_name IS NOT NULL THEN tivo_prev_station.inscape_station_name
                    WHEN LOWER(tivo_prev_station.station_affil) LIKE '%affiliate%'
                          OR LOWER(tivo_prev_station.station_affil) LIKE '%independent%'
                          OR LOWER(tivo_prev_station.station_affil) LIKE '%low power%' THEN tivo_prev_station.station_affil END
         WHEN prev_chanb.channel_name IS NOT NULL OR c.prev_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN 'OBFUSCATED'
         WHEN c.prev_vizio_epg_station IS NOT NULL THEN prev_vizio_station.name
    END AS tivo_prev_network_affiliate
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN tms_next_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.tms_next_station_id IS NOT NULL THEN
           CASE WHEN tms_next_station.inscape_station_name IS NOT NULL THEN tms_next_station.inscape_station_name
                WHEN LOWER(tms_next_station.station_affil) LIKE '%affiliate%'
                      OR LOWER(tms_next_station.station_affil) LIKE '%independent%'
                      OR LOWER(tms_next_station.station_affil) LIKE '%low power%' THEN tms_next_station.station_affil END
         WHEN next_chanb.channel_name IS NOT NULL OR c.next_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN 'OBFUSCATED'
         WHEN c.next_vizio_epg_station IS NOT NULL THEN next_vizio_station.name
    END AS tms_next_network_affiliate
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN tivo_next_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.next_station_id IS NOT NULL THEN
           CASE WHEN tivo_next_station.inscape_station_name IS NOT NULL THEN tivo_next_station.inscape_station_name
                WHEN LOWER(tivo_next_station.station_affil) LIKE '%affiliate%'
                      OR LOWER(tivo_next_station.station_affil) LIKE '%independent%'
                      OR LOWER(tivo_next_station.station_affil) LIKE '%low power%' THEN tivo_next_station.station_affil END
         WHEN next_chanb.channel_name IS NOT NULL OR c.next_vizio_epg_station IN ('98989898989898', '9898989898', '-1') THEN 'OBFUSCATED'
         WHEN c.next_vizio_epg_station IS NOT NULL THEN next_vizio_station.name
    END AS tivo_next_network_affiliate
  ---------------------------- Liveness -----------------------
  , CASE WHEN cl2.client_id IS NOT NULL THEN NULL
         WHEN tvis.category = 'APPS' and tis.app_name = 'OBFUSCATED' AND c.prev_vizio_epg_station IS NOT NULL THEN 't'
         ELSE CASE WHEN prev_content.is_live = TRUE THEN 't' WHEN prev_content.is_live = FALSE THEN 'f' END
    END AS live
  ---------------------------- App Service --------------------
  , CASE WHEN UPPER(tvis.category) = 'APPS' THEN
              CASE WHEN c.prev_vizio_epg_station IS NOT NULL THEN 'WatchFree+'
                   WHEN c.prev_vizio_epg_station IS NULL AND tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
                   WHEN LOWER(tis.app_name) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora','tv games')
                    AND COALESCE(c.prev_show_id, c.tms_prev_show_id) IS NOT NULL THEN NULL
                   WHEN LOWER(coalesce(tis.app_name)) = 'unknown' THEN NULL
                   ELSE tis.app_name END
         WHEN prev_content.is_live = true
              AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4','playstation 5','roku') THEN 'vMVPD'
  END AS app_service
  ------------ Prev and Next Session Times -------------------
  , c.prev_session_start AS prev_ts_start
  , c.prev_session_end AS prev_ts_end
  , c.next_session_start AS next_ts_start
  , c.next_session_end AS next_ts_end
  ------------ IP Address, Input Category/Device -------------
  , ip.ip_address AS ip
  , tvis.category AS input_category
  , tvis.input_device AS input_device
  ----------------- Comm Metadata ----------------------------
  , REPLACE(m.brand_name, ',', '') AS brand_name
  , REPLACE(m.title, ',', '') AS title
  , m.duration AS duration
  , c.fk_commercial_source_id
  ------------------ Conditions -------------------
  --------------------- Bools ---------------------
  , CASE WHEN COALESCE(tms_prev_nielsen_blacklist.station_id, tivo_prev_nielsen_blacklist.station_id) IS NOT NULL
          AND (COALESCE(tms_rep_local_prev.station_id, tms_rep_nyc_nat_prev.station_id,
                        tivo_rep_local_prev.station_id, tivo_rep_nyc_nat_prev.station_id) IS NULL
               OR COALESCE(tms_prev_nielsen_blacklist.ingest_time, tivo_prev_nielsen_blacklist.ingest_time) IS NOT NULL) THEN TRUE
         ELSE FALSE
    END AS prev_nielsen_exclusive
  , CASE WHEN COALESCE(tms_next_nielsen_blacklist.station_id, tivo_next_nielsen_blacklist.station_id) IS NOT NULL
         AND (COALESCE(tms_rep_local_next.station_id, tms_rep_nyc_nat_next.station_id,
                       tivo_rep_local_next.station_id, tivo_rep_nyc_nat_next.station_id) IS NULL
              OR COALESCE(tms_next_nielsen_blacklist.ingest_time, tivo_next_nielsen_blacklist.ingest_time) IS NOT NULL) THEN TRUE
         ELSE FALSE
    END AS next_nielsen_exclusive
  ------------------ Later Aggs ------------------
  , CASE WHEN UPPER(tvis.category) = 'APPS'
              AND prev_vizio_station.name IS NULL
              AND acrb.app_name IS NOT NULL THEN CASE WHEN acrb.client_name IS NULL THEN 'ALL'
                                                      ELSE acrb.client_name END
    END AS acrb_client
  , CASE WHEN UPPER(tvis.category) = 'APPS' THEN
          CASE WHEN c.prev_vizio_epg_station IS NOT NULL THEN NULL
               WHEN c.prev_vizio_epg_station IS NULL AND tis.app_name = 'WatchFree+' THEN NULL
               WHEN appb.app_name IS NOT NULL THEN CASE WHEN appb.client_name IS NULL THEN 'ALL'
                                                        ELSE appb.client_name END
          END
    END AS appb_client
  , CASE WHEN COALESCE(tivo_prev_station_blacklist.station_id, tms_prev_station_blacklist.station_id) IS NOT NULL THEN
              CASE WHEN COALESCE(tivo_prev_station_blacklist.client_name, tms_prev_station_blacklist.client_name) IS NULL THEN 'ALL'
                   ELSE COALESCE(tivo_prev_station_blacklist.client_name, tms_prev_station_blacklist.client_name) END
    END AS prev_station_blacklist_client
  , CASE WHEN COALESCE(tivo_next_station_blacklist.station_id, tms_next_station_blacklist.station_id) IS NOT NULL THEN
              CASE WHEN COALESCE(tivo_next_station_blacklist.client_name, tms_next_station_blacklist.client_name) IS NULL THEN 'ALL'
                   ELSE COALESCE(tivo_next_station_blacklist.client_name, tms_next_station_blacklist.client_name) END
    END AS next_station_blacklist_client
  -----------------------------------------------------------------
  FROM prod.detection.viewing_commercials_firehose_dedup_cfe_merge AS c
  -- Joins that do not need to be modified
  JOIN prod.detection.zoo AS z
    ON c.fk_zoo_id = z.zoo_id
   AND z.zoo = 'control-zoo-dtsprod.tvinteractive.tv'
  INNER JOIN prod.detection.tv AS tv
    ON c.fk_tvid = tv.tvid
   AND tv.oem = 'VIZIO'
  JOIN prod.detection.tv_populations AS tp
    ON c.fk_tvid = tp.fk_tvid
  JOIN prod.detection.populations AS pop
    ON tp.fk_population_id = pop.population_id
   AND LOWER(pop.population_name) = 'opted_in'
  JOIN prod.detection.tv_settings AS tv_settings
    ON c.session_start < tv_settings.next_create_timestamp
   AND c.session_start >= tv_settings.create_timestamp
   AND c.fk_tvid = tv_settings.fk_tvid
   AND tv_settings.create_timestamp <= '{end_date}'::timestamp
   AND tv_settings.next_create_timestamp >= '{start_date}'::timestamp
  JOIN prod.detection.settings AS settings
    ON tv_settings.fk_settings_id = settings.settings_id
   AND UPPER(settings.country_name) = 'USA'
  ----------------------Location----------------------
  JOIN prod.detection.location AS location
    ON c.fk_location_id = location.location_id
   AND UPPER(location.country_code) = 'US'
  LEFT OUTER JOIN prod.detection.dma AS dma
    ON c.fk_dma_id = dma.dma_id
  ----------------------Input Joins----------------------
  JOIN prod.detection.tv_input_stats_firehose tvis
    ON c.session_start >= tvis.create_timestamp
   AND c.session_start < tvis.next_create_timestamp
   AND tvis.create_timestamp <= '{end_date}'::timestamp
   AND tvis.next_create_timestamp >= '{start_date}'::timestamp
   AND c.fk_tvid = tvis.fk_tvid
   AND c.fk_input_source_id = tvis.fk_input_source_id
  LEFT OUTER JOIN prod.detection.tv_inputsource tis
    ON c.session_start >=  (tis.create_timestamp::double)::timestamp
   AND c.session_start <  (tis.next_create_timestamp::double)::timestamp
   AND tis.create_timestamp <= ('{end_date}'::timestamp::double)::timestamp
   AND tis.next_create_timestamp >= ('{start_date}'::timestamp::double)::timestamp
   AND c.fk_tvid = tis.fk_tvid
   AND c.fk_input_source_id = tis.fk_input_source_id
  ----------------------Prev Content----------------------
  LEFT OUTER JOIN viewing_content_firehose AS prev_content
    ON c.fk_tvid = prev_content.fk_tvid
   AND prev_content.session_start = c.prev_session_start
  ----------------------IP Address----------------------
  LEFT OUTER JOIN prod.detection.tv_ip_address AS ip
    ON c.session_start >= ip.create_timestamp
   AND c.session_start < ip.next_create_timestamp
   AND ip.create_timestamp <= '{end_date}'::timestamp
   AND ip.next_create_timestamp >= '{start_date}'::timestamp
   AND c.fk_tvid = ip.fk_tvid
  ------------------Commercials Metadata----------------------
  INNER JOIN new_comm_id_ext_firehose m
    ON c.external_id = m.external_id
  --------------- WF+ Prev/Next Metadata Joins ---------------
  LEFT OUTER JOIN prod.detection.vizio_epg_station AS prev_vizio_station
    ON TRY_CAST(c.prev_vizio_epg_station AS STRING) <=> TRY_CAST(prev_vizio_station.station_id AS STRING)
  LEFT OUTER JOIN epg_program_aggregate AS prev_vizio_program
    ON TRY_CAST(c.prev_vizio_epg_program AS STRING) <=> TRY_CAST(prev_vizio_program.program_aggregate_id AS STRING)
   AND TRY_CAST(c.prev_vizio_epg_program AS STRING) NOT IN ('0', '', '-1')
   AND TRY_CAST(c.prev_vizio_epg_program AS STRING) IS NOT NULL
  LEFT OUTER JOIN prod.detection.vizio_epg_station AS next_vizio_station
    ON TRY_CAST(c.next_vizio_epg_station AS STRING) <=> TRY_CAST(next_vizio_station.station_id AS STRING)
  LEFT OUTER JOIN epg_program_aggregate AS next_vizio_program
    ON TRY_CAST(c.next_vizio_epg_program AS STRING) <=> TRY_CAST(next_vizio_program.program_aggregate_id AS STRING)
   AND TRY_CAST(c.next_vizio_epg_program AS STRING) NOT IN ('0', '', '-1')
   AND TRY_CAST(c.next_vizio_epg_program AS STRING) IS NOT NULL
  ------------------TiVo/TMS Metadata------------------
  LEFT OUTER JOIN prod.detection.epg_station AS tivo_prev_station
    ON tivo_prev_station.station_id = c.prev_station_id
   AND tivo_prev_station.vendor_name = 'TIVO'
  LEFT OUTER JOIN inscape_map_deduped AS tivo_prev_map
    ON tivo_prev_map.mapped_vendor_station_id = c.prev_station_id
   AND tivo_prev_map.mapped_vendor = 'TIVO'
  LEFT OUTER JOIN prod.detection.epg_show AS tivo_prev_program
    ON tivo_prev_program.show_id = c.prev_show_id
   AND tivo_prev_program.vendor_name = 'TIVO'

  LEFT OUTER JOIN prod.detection.epg_station AS tms_prev_station
    ON tms_prev_station.station_id = c.tms_prev_station_id
   AND tms_prev_station.vendor_name = 'TMS'
  LEFT OUTER JOIN inscape_map_deduped AS tms_prev_map
    ON tms_prev_map.mapped_vendor_station_id = c.tms_prev_station_id
   AND tms_prev_map.mapped_vendor = 'TMS'
  LEFT OUTER JOIN prod.detection.epg_show AS tms_prev_program
    ON tms_prev_program.show_id = c.tms_prev_show_id
   AND tms_prev_program.vendor_name = 'TMS'

  LEFT OUTER JOIN prod.detection.epg_station AS tivo_next_station
    ON tivo_next_station.station_id = c.next_station_id
   AND tivo_next_station.vendor_name = 'TIVO'
  LEFT OUTER JOIN inscape_map_deduped AS tivo_next_map
    ON tivo_next_map.mapped_vendor_station_id = c.next_station_id
   AND tivo_next_map.mapped_vendor = 'TIVO'
  LEFT OUTER JOIN prod.detection.epg_show AS tivo_next_program
    ON tivo_next_program.show_id = c.next_show_id
   AND tivo_next_program.vendor_name = 'TIVO'

  LEFT OUTER JOIN prod.detection.epg_station AS tms_next_station
    ON tms_next_station.station_id = c.tms_next_station_id
   AND tms_next_station.vendor_name = 'TMS'
  LEFT OUTER JOIN inscape_map_deduped AS tms_next_map
    ON tms_next_map.mapped_vendor_station_id = c.tms_next_station_id
   AND tms_next_map.mapped_vendor = 'TMS'
  LEFT OUTER JOIN prod.detection.epg_show AS tms_next_program
    ON tms_next_program.show_id = c.tms_next_show_id
   AND tms_next_program.vendor_name = 'TMS'
  ----------------------File Content Joins----------------------
  LEFT OUTER JOIN prod.detection.content_id_external_firehose AS prev_filecontent
    ON c.prev_show_id = prev_filecontent.fk_content_id
  LEFT OUTER JOIN prod.detection.content_id_external_firehose AS next_filecontent
    ON c.next_show_id = next_filecontent.fk_content_id
  LEFT OUTER JOIN prod.detection.content_id_external_firehose AS m_filter
    ON m_filter.fk_content_id = prev_content.fk_content_id
  LEFT OUTER JOIN content_ids_firehose as prev_cid
    ON c.prev_show_id = prev_cid.content_id
  LEFT OUTER JOIN content_ids_firehose as next_cid
    ON c.next_show_id = next_cid.content_id
  LEFT OUTER JOIN prod.detection.clients cl2
    ON m_filter.fk_client_id = cl2.client_id
   AND cl2.client_name NOT IN ('kinetiq', 'SpringServe-Prod')
  ----------------------Station Blacklist----------------------
  LEFT OUTER JOIN station_distribution_blacklist AS tivo_prev_station_blacklist
    ON tivo_prev_station_blacklist.station_id = c.prev_station_id
  LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS tivo_prev_station_obfs
    ON tivo_prev_station_obfs.vendor_station_id = c.prev_station_id
  LEFT OUTER JOIN station_distribution_blacklist AS tivo_next_station_blacklist
    ON tivo_next_station_blacklist.station_id = c.next_station_id
  LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS tivo_next_station_obfs
    ON tivo_next_station_obfs.vendor_station_id = c.next_station_id

  LEFT OUTER JOIN station_distribution_blacklist AS tms_prev_station_blacklist
    ON tms_prev_station_blacklist.station_id = c.tms_prev_station_id
  LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS tms_prev_station_obfs
    ON tms_prev_station_obfs.vendor_station_id = c.tms_prev_station_id
  LEFT OUTER JOIN station_distribution_blacklist AS tms_next_station_blacklist
    ON tms_next_station_blacklist.station_id = c.tms_next_station_id
  LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS tms_next_station_obfs
    ON tms_next_station_obfs.vendor_station_id = c.tms_next_station_id
  -----------------App + WF+ Station Blacklist-----------------
  LEFT OUTER JOIN activity_obfuscation appb
    ON tis.app_name = appb.app_name
  LEFT OUTER JOIN viewing_obfuscation AS acrb
    ON tis.app_name = acrb.app_name
  LEFT OUTER JOIN prod.detection.free_channels_distribution_blacklist prev_chanb
    ON prev_vizio_station.name = prev_chanb.channel_name
  LEFT OUTER JOIN prod.detection.free_channels_distribution_blacklist next_chanb
    ON next_vizio_station.name = next_chanb.channel_name
  -----------------Nielsen Blacklist-----------------
  LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS tivo_prev_nielsen_blacklist
    ON tivo_prev_map.inscape_station_id = tivo_prev_nielsen_blacklist.station_id
   AND c.session_start >= tivo_prev_nielsen_blacklist.blacklist_start
   AND c.session_start < tivo_prev_nielsen_blacklist.blacklist_end
  LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS tms_prev_nielsen_blacklist
    ON tms_prev_map.inscape_station_id = tms_prev_nielsen_blacklist.station_id
   AND c.session_start >= tms_prev_nielsen_blacklist.blacklist_start
   AND c.session_start < tms_prev_nielsen_blacklist.blacklist_end

  LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS tivo_next_nielsen_blacklist
    ON tivo_next_map.inscape_station_id = tivo_next_nielsen_blacklist.station_id
   AND c.session_start >= tivo_next_nielsen_blacklist.blacklist_start
   AND c.session_start < tivo_next_nielsen_blacklist.blacklist_end
  LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS tms_next_nielsen_blacklist
    ON tms_next_map.inscape_station_id = tms_next_nielsen_blacklist.station_id
   AND c.session_start >= tms_next_nielsen_blacklist.blacklist_start
   AND c.session_start < tms_next_nielsen_blacklist.blacklist_end

  LEFT OUTER JOIN nielsen_replacement_local_alias AS tivo_rep_local_prev
    ON tivo_prev_map.inscape_station_id = tivo_rep_local_prev.station_id
   AND c.prev_show_id = tivo_rep_local_prev.fk_show_id
   AND c.fk_dma_id = tivo_rep_local_prev.dma_id
  LEFT OUTER JOIN nielsen_replacement_local_alias AS tms_rep_local_prev
    ON tms_prev_map.inscape_station_id = tms_rep_local_prev.station_id
   AND c.tms_prev_show_id = tms_rep_local_prev.fk_show_id
   AND c.fk_dma_id = tms_rep_local_prev.dma_id

  LEFT OUTER JOIN nielsen_replacement_national_nyc_alias AS tivo_rep_nyc_nat_prev
    ON tivo_prev_map.inscape_station_id = tivo_rep_nyc_nat_prev.station_id
   AND c.prev_show_id = tivo_rep_nyc_nat_prev.fk_show_id
  LEFT OUTER JOIN nielsen_replacement_national_nyc_alias AS tms_rep_nyc_nat_prev
    ON tms_prev_map.inscape_station_id = tms_rep_nyc_nat_prev.station_id
   AND c.tms_prev_show_id = tms_rep_nyc_nat_prev.fk_show_id

  LEFT OUTER JOIN nielsen_replacement_local_alias AS tivo_rep_local_next
    ON tivo_next_map.inscape_station_id = tivo_rep_local_next.station_id
   AND c.next_show_id = tivo_rep_local_next.fk_show_id
   AND c.fk_dma_id = tivo_rep_local_next.dma_id
  LEFT OUTER JOIN nielsen_replacement_local_alias AS tms_rep_local_next
    ON tms_next_map.inscape_station_id = tms_rep_local_next.station_id
   AND c.tms_next_show_id = tms_rep_local_next.fk_show_id
   AND c.fk_dma_id = tms_rep_local_next.dma_id

  LEFT OUTER JOIN nielsen_replacement_national_nyc_alias AS tivo_rep_nyc_nat_next
    ON tivo_next_map.inscape_station_id = tivo_rep_nyc_nat_next.station_id
   AND c.next_show_id = tivo_rep_nyc_nat_next.fk_show_id
  LEFT OUTER JOIN nielsen_replacement_national_nyc_alias AS tms_rep_nyc_nat_next
    ON tms_next_map.inscape_station_id = tms_rep_nyc_nat_next.station_id
   AND c.tms_next_show_id = tms_rep_nyc_nat_next.fk_show_id
  WHERE c.session_start >= '{start_date}'::timestamp
    AND c.session_start < '{end_date}'::timestamp
    AND c.partition_key >= '{start_date}'::timestamp::DATE
    AND c.partition_key <= '{end_date}'::timestamp::DATE
)
GROUP BY tvid, fk_tvid, zipcode, dma, external_id, mt_start, session_start, session_end
, tms_prev_episode_id, tivo_prev_episode_id, tms_next_episode_id, tivo_next_episode_id
, tms_prev_show_title, tivo_prev_show_title, tms_next_show_title, tivo_next_show_title
, tms_prev_callsign, tivo_prev_callsign, tms_next_callsign, tivo_next_callsign
, tms_prev_network_affiliate, tivo_prev_network_affiliate, tms_next_network_affiliate, tivo_next_network_affiliate
, live, app_service, prev_ts_start, prev_ts_end, next_ts_start, next_ts_end
, ip, input_category, input_device, brand_name, title, duration
, fk_commercial_source_id, prev_nielsen_exclusive, next_nielsen_exclusive;

In [0]:
SELECT c.tms_next_station_id = 0, c.next_station_id = 0, COUNT(*)
FROM prod.detection.viewing_commercials_firehose c
WHERE c.session_start >= '2025-01-22T07:00:00'::timestamp
  AND c.session_start < '2025-01-22T08:00:00'::timestamp
  AND c.partition_key >= '2025-01-22T07:00:00'::timestamp::DATE
  AND c.partition_key <= '2025-01-22T08:00:00'::timestamp::DATE
GROUP BY 1, 2

In [0]:
SELECT current_timestamp() - INTERVAL '12 HOURS'